In [1]:
import pandas as pd
import numpy as np

In [2]:
adni_df = pd.read_csv('adniclean.csv')
meta_df = pd.read_csv('metaclean.csv')
print(f"ADNI data: {len(adni_df)} rows, {len(adni_df.columns)} columns")
print(f"Meta data: {len(meta_df)} rows, {len(meta_df.columns)} columns")

ADNI data: 10196 rows, 5 columns
Meta data: 4228 rows, 106 columns


In [3]:
#removing any whitespace
adni_df.columns = adni_df.columns.str.strip()
meta_df.columns = meta_df.columns.str.strip()

In [4]:
#check if RID and VISCODE columns exist
required_cols = ['RID', 'VISCODE']
for col in required_cols:
    if col not in adni_df.columns:
        print(f"ERROR: Column '{col}' not found in ADNI file")
    if col not in meta_df.columns:
        print(f"ERROR: Column '{col}' not found in Meta file")

In [5]:
#remove rows with missing data
adni_df = adni_df.dropna(subset=['RID', 'VISCODE'])
meta_df = meta_df.dropna(subset=['RID', 'VISCODE'])

In [6]:
print(f"After removing missing RID/VISCODE:")
print(f"ADNI: {len(adni_df)} rows")
print(f"Meta: {len(meta_df)} rows")

After removing missing RID/VISCODE:
ADNI: 10196 rows
Meta: 4228 rows


In [7]:
#identify common patients in both files
adni_patients = set(adni_df['RID'].unique())
meta_patients = set(meta_df['RID'].unique())
common_patients = adni_patients.intersection(meta_patients)

In [8]:
print(f"ADNI unique patients: {len(adni_patients)}")
print(f"Meta unique patients: {len(meta_patients)}")
print(f"Common patients: {len(common_patients)}")

ADNI unique patients: 2020
Meta unique patients: 1274
Common patients: 1274


In [9]:
#filter to only common patients and create matching keys for merging later
adni_df = adni_df[adni_df['RID'].isin(common_patients)]
meta_df = meta_df[meta_df['RID'].isin(common_patients)]

adni_df['key'] = adni_df['RID'].astype(str) + '_' + adni_df['VISCODE'].astype(str)
meta_df['key'] = meta_df['RID'].astype(str) + '_' + meta_df['VISCODE'].astype(str)

In [10]:
#identify matching RID-VISCODE combinations
adni_keys = set(adni_df['key'])
meta_keys = set(meta_df['key'])
matching_keys = adni_keys.intersection(meta_keys)

In [11]:
print(f"ADNI RID-VISCODE combinations: {len(adni_keys)}")
print(f"Meta RID-VISCODE combinations: {len(meta_keys)}")
print(f"Matching combinations: {len(matching_keys)}")

ADNI RID-VISCODE combinations: 8341
Meta RID-VISCODE combinations: 4189
Matching combinations: 4187


In [12]:
#new filtered df with only matching combinations
adni_filtered = adni_df[adni_df['key'].isin(matching_keys)]
meta_filtered = meta_df[meta_df['key'].isin(matching_keys)]

In [13]:
#merge
merged_df = pd.merge(
    adni_filtered, 
    meta_filtered, 
    on=['RID', 'VISCODE'], 
    how='inner',
    suffixes=('_adni', '_meta')
)

In [14]:
#drop temporary key columns
merged_df = merged_df.drop(['key_adni', 'key_meta'], axis=1)

In [15]:
#sanity check
unique_patients_final = len(merged_df['RID'].unique())
visits_per_patient = merged_df.groupby('RID').size()
patients_with_multiple_visits = len(visits_per_patient[visits_per_patient > 1])

print(f"Final merged dataset:")
print(f"Total records: {len(merged_df)}")
print(f"Unique patients: {unique_patients_final}")
print(f"Patients with multiple visits: {patients_with_multiple_visits}")
print(f"Average visits per patient: {visits_per_patient.mean():.1f}")

Final merged dataset:
Total records: 4226
Unique patients: 1274
Patients with multiple visits: 1236
Average visits per patient: 3.3


In [16]:
merged_df.head(20)

,RID,VISCODE,DX_bl,DX,Month,L_HISTIDINE,L_LYSINE,L_SERINE,BETA_ALANINE,L_ALANINE,...,NORCA,NORDCA,TCA,TCDCA,TDCA,TUDCA,TAMCA,UCA,UDCA,BUDCA
0,3,bl,Dementia,Dementia,0,99.4952,261.459,102.6600,7.20199,512.990,...,16.97580,2.93506,55.71210,154.11700,175.796000,1.7,11.3,0.818550,15.6,25.1
1,3,m12,Dementia,Dementia,12,88.3782,222.734,85.2418,8.57907,329.971,...,10.92950,3.96786,15.21140,27.90280,32.562200,1.1,6.1,0.389794,9.1,14.4
2,3,m24,Dementia,Dementia,24,91.1270,209.408,89.8045,5.11167,330.171,...,13.04350,1.81410,24.26470,60.03560,46.066700,1.0,4.0,0.786292,1.7,6.7
3,4,bl,LMCI,MCI,0,78.1877,208.347,95.9779,3.65515,378.315,...,4.12863,2.05665,15.22970,97.38330,26.792700,2.1,1.6,0.225801,24.5,216.8
4,4,m12,LMCI,MCI,12,82.9090,201.343,105.9390,3.01537,318.439,...,2.93855,3.92328,13.02320,45.45310,16.016200,2.5,1.7,0.969108,67.4,197.2
5,5,bl,CN,CN,0,95.3085,185.013,134.3330,5.38564,675.853,...,11.09640,1.30000,349.33300,501.46500,455.264000,11.1,13.3,1.239740,86.8,224.3
6,5,m12,CN,CN,12,91.0173,216.346,115.3330,5.63351,564.394,...,10.17870,2.21889,190.72900,155.67200,137.876000,3.0,8.1,0.672374,2.9,34.8
7,5,m24,CN,CN,24,79.7514,226.186,111.8880,2.96023,537.927,...,8.36961,1.75664,63.07130,80.81000,85.708800,2.1,4.8,0.901360,12.8,137.7
8,6,bl,LMCI,MCI,0,89.7745,297.523,147.6870,4.34735,514.569,...,3.43190,3.59681,16.82620,75.07010,13.342100,1.6,1.3,0.991451,13.1,19.1
9,6,m12,LMCI,MCI,12,64.7752,178.735,192.9330,2.25040,610.721,...,3.56488,4.42993,34.63120,75.40070,29.867400,4.3,10.7,1.119740,7.4,46.5


In [17]:
merged_df.to_csv('merged_data.csv', index=False)